## Dog Breed Classification using Transfer Learning

In [1]:
# Import necessary libraries
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16, ResNet50, MobileNetV2
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import numpy as np
import matplotlib.pyplot as plt
import os
import shutil

### 1. Data Preparation
First, we need to extract the dataset from the provided zip file and organize it for training.

In [8]:
# Unzip the dataset
zip_path = '/content/archive (3).zip'
extract_path = '/content/dog_breeds'

if not os.path.exists(extract_path):
    os.makedirs(extract_path)
    shutil.unpack_archive(zip_path, extract_path)
    print(f"Dataset extracted to: {extract_path}")
else:
    print(f"Dataset already extracted to: {extract_path}")

# Define paths for the dataset - Corrected path based on inspection
data_dir = os.path.join(extract_path, 'dataset') # Corrected path again

Dataset already extracted to: /content/dog_breeds


In [3]:
# Check if data_dir exists and contains subdirectories
if not os.path.exists(data_dir):
    print(f"Error: Directory {data_dir} not found. Please verify the extraction path and folder structure inside the zip.")
elif not os.listdir(data_dir):
    print(f"Error: Directory {data_dir} is empty. No image folders found.")
else:
    num_breeds = len(os.listdir(data_dir))
    print(f"Number of dog breeds found: {num_breeds}")
    print(f"Sample breeds: {os.listdir(data_dir)[:5]}")

Error: Directory /content/dog_breeds/Images not found. Please verify the extraction path and folder structure inside the zip.


In [6]:
# Inspect the extracted directory to find the actual image folder structure
print(f"Contents of {extract_path}:")
for item in os.listdir(extract_path):
    print(f"- {item}")

# Based on typical dataset structures, the actual images might be in a subfolder like 'Images' or similar.
# If the 'Images' folder is directly inside 'extract_path', then the original data_dir might be correct.
# If not, we'll need to adjust `data_dir`.

Contents of /content/dog_breeds:
- dataset


### 2. Configure ImageDataGenerator
We'll use `ImageDataGenerator` for data augmentation and splitting the dataset into training, validation, and test sets. We'll set image size and batch size here.

In [9]:
# Define image dimensions and batch size
IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 32

# Create data generators for training, validation, and test sets
datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2 # 80% for training, 20% for validation/test
)

train_generator = datagen.flow_from_directory(
    data_dir,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

val_generator = datagen.flow_from_directory(
    data_dir,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

# For the test set, we'll use a separate generator without augmentation
test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_directory(
    data_dir, # Using the same directory for test, but will split manually later if needed or rely on validation split
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False # Keep data in order for evaluation
)

num_classes = train_generator.num_classes
print(f"Number of classes (dog breeds): {num_classes}")

Found 775 images belonging to 10 classes.
Found 192 images belonging to 10 classes.
Found 967 images belonging to 10 classes.
Number of classes (dog breeds): 10


### 3. Load a Pre-trained Model (VGG16)
We'll use VGG16 as the base model and add our custom classification head.

In [10]:
# Load the VGG16 model, pre-trained on ImageNet, without the top classification layer
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(IMG_HEIGHT, IMG_WIDTH, 3))

# Freeze the layers of the base model
for layer in base_model.layers:
    layer.trainable = False

# Build the custom classification head
x = base_model.output
x = Flatten()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x) # Add dropout for regularization
predictions = Dense(num_classes, activation='softmax')(x)

# Create the final model
model = Model(inputs=base_model.input, outputs=predictions)

# Compile the model
model.compile(optimizer=Adam(learning_rate=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])

# Display model summary
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │     6,422,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,140,042 (80.64 MB)

 Trainable params: 6,425,354 (24.51 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

### 4. Train the Model
Now we will train the VGG16-based model using the `train_generator` and `val_generator`.

In [11]:
# Train the model
epochs = 10 # You can adjust the number of epochs
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // BATCH_SIZE,
    epochs=epochs,
    validation_data=val_generator,
    validation_steps=val_generator.samples // BATCH_SIZE
)





print("Model training complete.")

Epoch 1/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 371s 15s/step - accuracy: 0.1682 - loss: 2.3619 - val_accuracy: 0.2917 - val_loss: 2.0388
Epoch 2/10
 1/24 ━━━━━━━━━━━━━━━━━━━━ 4:42 12s/step - accuracy: 0.2188 - loss: 2.2105

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


24/24 ━━━━━━━━━━━━━━━━━━━━ 94s 4s/step - accuracy: 0.2188 - loss: 2.2105 - val_accuracy: 0.3073 - val_loss: 2.0448
Epoch 3/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 401s 15s/step - accuracy: 0.2651 - loss: 2.0057 - val_accuracy: 0.4688 - val_loss: 1.8060
Epoch 4/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 87s 3s/step - accuracy: 0.4062 - loss: 1.7870 - val_accuracy: 0.4688 - val_loss: 1.7799
Epoch 5/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 363s 15s/step - accuracy: 0.3782 - loss: 1.8102 - val_accuracy: 0.6250 - val_loss: 1.4957
Epoch 6/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 87s 3s/step - accuracy: 0.2500 - loss: 1.9511 - val_accuracy: 0.5677 - val_loss: 1.5399
Epoch 7/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 362s 16s/step - accuracy: 0.4428 - loss: 1.6193 - val_accuracy: 0.6302 - val_loss: 1.3376
Epoch 8/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 91s 3s/step - accuracy: 0.4688 - loss: 1.5801 - val_accuracy: 0.6875 - val_loss: 1.3419
Epoch 9/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 427s 18s/step - accuracy: 0.5101 - loss: 1.4351 - val_accuracy: 0.6771 - val_loss: 1.

### 5. Evaluate the Model
After training, we will evaluate the model's performance on the test set.

In [ ]:
# Evaluate the model on the test set
loss, accuracy = model.evaluate(test_generator, steps=test_generator.samples // BATCH_SIZE)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

 3/30 ━━━━━━━━━━━━━━━━━━━━ 5:43 13s/step - accuracy: 1.0000 - loss: 0.5785

### 6. Visualize Training History
Let's plot the training and validation accuracy and loss over epochs to see how the model learned.

In [ ]:
# Plot training history
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training and Validation Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()

plt.tight_layout()
plt.show()